In [15]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CENSUS_DIR = DATA_DIR / "census" / "processed"

JOBS_DIR = DATA_DIR / "jobs" / "lodes_processed"

jobs_bgs_path = JOBS_DIR / "lodes_jobs_2022_with_geometry.gpkg"
tracts_path = CENSUS_DIR / "tracts_clean.gpkg"

print("jobs_bgs_path:", jobs_bgs_path)
print("tracts_path:", tracts_path)

jobs_bgs_path: c:\Users\arjav\DevSource\toc-performance-dashboard\data\jobs\lodes_processed\lodes_jobs_2022_with_geometry.gpkg
tracts_path: c:\Users\arjav\DevSource\toc-performance-dashboard\data\census\processed\tracts_clean.gpkg


In [16]:
bg_jobs = gpd.read_file(jobs_bgs_path)
bg_jobs = bg_jobs.to_crs(2868)

bg_jobs.columns

Index(['GEOID', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE', 'C000', 'CE01',
       'CE02', 'CE03', 'area_sq_ft', 'area_acres', 'jobs_total',
       'jobs_per_acre', 'geometry'],
      dtype='object')

In [17]:
tracts = gpd.read_file(tracts_path)
tracts = tracts.to_crs(2868)

tracts.columns

Index(['GISJOIN', 'geometry'], dtype='object')

In [18]:
tracts["GISJOIN"].head()

0    G0400130010102
1    G0400130010103
2    G0400130010104
3    G0400130030401
4    G0400130030402
Name: GISJOIN, dtype: object

In [19]:
bg_jobs["GEOID"].head()

0    040130101021
1    040130101021
2    040130101021
3    040130101021
4    040130101021
Name: GEOID, dtype: object

In [20]:
# first we have to slice the GISJOIN to align with GEOID

tracts["TRACT_GEOID"] = tracts["GISJOIN"].str.slice(1, 12+1)
bg_jobs["BG_GEOID"] = bg_jobs["GEOID"].astype(str)

In [21]:
bg_x_tract = gpd.overlay(
    bg_jobs,
    tracts,
    how="intersection",
    keep_geom_type=False)

len(bg_x_tract)

12890

In [22]:
# compute intersection area in sqft, then convert to acres

bg_x_tract["intersect_sqft"] = bg_x_tract.geometry.area
bg_x_tract["intersect_acres"] = bg_x_tract["intersect_sqft"] / 43560

bg_x_tract.head()

,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,C000,CE01,CE02,CE03,area_sq_ft,area_acres,jobs_total,jobs_per_acre,BG_GEOID,GISJOIN,TRACT_GEOID,geometry,intersect_sqft,intersect_acres
0,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,593324.922484,455,0.000767,040130101021,G0400130010102,040013001010,"POLYGON ((827865.552 1091578.775, 828791.756 1...",2.584523e+10,593324.922484
1,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,593324.922484,455,0.000767,040130101021,G0400130010103,040013001010,"MULTILINESTRING ((765755.318 1002785.693, 7657...",0.000000e+00,0.000000
2,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,593324.922484,455,0.000767,040130101021,G0400130010104,040013001010,"MULTILINESTRING ((779686.839 987205.074, 77974...",0.000000e+00,0.000000
3,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,593324.922484,455,0.000767,040130101021,G0400130216829,040013021682,"MULTILINESTRING ((739426.916 1023686.255, 7394...",0.000000e+00,0.000000
4,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,593324.922484,455,0.000767,040130101021,G0400130420107,040013042010,"MULTILINESTRING ((781319.453 907790.761, 78015...",0.000000e+00,0.000000


In [23]:
# distribute BG jobs into tracts proportionally by area

bg_x_tract["area_weight"] = (
    bg_x_tract["intersect_sqft"] /
    bg_x_tract["area_sq_ft"]
)

# sanity check

bg_x_tract["area_weight"].describe()

count    12890.000000
mean         0.217688
std          0.412690
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: area_weight, dtype: float64

In [24]:
# distribute jobs

bg_x_tract["jobs_contrib"] = (
    bg_x_tract["jobs_total"].fillna(0) *
    bg_x_tract["area_weight"]
)

bg_x_tract.head()

,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,C000,CE01,CE02,CE03,area_sq_ft,...,jobs_total,jobs_per_acre,BG_GEOID,GISJOIN,TRACT_GEOID,geometry,intersect_sqft,intersect_acres,area_weight,jobs_contrib
0,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,...,455,0.000767,040130101021,G0400130010102,040013001010,"POLYGON ((827865.552 1091578.775, 828791.756 1...",2.584523e+10,593324.922484,1.0,455.0
1,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,...,455,0.000767,040130101021,G0400130010103,040013001010,"MULTILINESTRING ((765755.318 1002785.693, 7657...",0.000000e+00,0.000000,0.0,0.0
2,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,...,455,0.000767,040130101021,G0400130010104,040013001010,"MULTILINESTRING ((779686.839 987205.074, 77974...",0.000000e+00,0.000000,0.0,0.0
3,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,...,455,0.000767,040130101021,G0400130216829,040013021682,"MULTILINESTRING ((739426.916 1023686.255, 7394...",0.000000e+00,0.000000,0.0,0.0
4,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,...,455,0.000767,040130101021,G0400130420107,040013042010,"MULTILINESTRING ((781319.453 907790.761, 78015...",0.000000e+00,0.000000,0.0,0.0


In [25]:
# and now, we aggregate to tract level

tract_jobs = (
    bg_x_tract
    .groupby("TRACT_GEOID", as_index=False)
    .agg({
        "jobs_contrib": "sum",
        "intersect_acres": "sum"
    })
)

# rename for clarity:

tract_jobs = tract_jobs.rename(columns={
    "jobs_contrib": "jobs_total",
    "intersect_acres": "acres_contributing"
})

tract_jobs.head()

,TRACT_GEOID,jobs_total,acres_contributing
0,040013001010,3640.0,690331.130975
1,040013003040,2730.0,20964.266666
2,040013004050,4594.0,8506.212017
3,040013004051,5547.0,655010.794056
4,040013004052,25300.0,7775.951501


In [26]:
# join!

tracts_with_jobs = tracts.merge(
    tract_jobs,
    left_on="TRACT_GEOID",
    right_on="TRACT_GEOID",
    how="left"
)

In [27]:
tracts_with_jobs["tract_acres"] = tracts_with_jobs.geometry.area / 43560

tracts_with_jobs.head()

,GISJOIN,geometry,TRACT_GEOID,jobs_total,acres_contributing,tract_acres
0,G0400130010102,"POLYGON ((827801.708 1091626.91, 827865.552 10...",040013001010,3640.0,690331.130975,676962.212041
1,G0400130010103,"POLYGON ((765755.318 1002785.693, 765592.367 1...",040013001010,3640.0,690331.130975,9303.984075
2,G0400130010104,"POLYGON ((765755.318 1002785.693, 765757.331 1...",040013001010,3640.0,690331.130975,4064.934859
3,G0400130030401,"POLYGON ((707726.563 1037891.22, 707726.326 10...",040013003040,2730.0,20964.266666,6558.993303
4,G0400130030402,"POLYGON ((707726.563 1037891.22, 707707.434 10...",040013003040,2730.0,20964.266666,14405.273363


In [28]:
# now we can compute jobs per acre

tracts_with_jobs["jobs_per_acre"] = (
    tracts_with_jobs["jobs_total"] /
    tracts_with_jobs["tract_acres"]
)

tracts_with_jobs.head()

,GISJOIN,geometry,TRACT_GEOID,jobs_total,acres_contributing,tract_acres,jobs_per_acre
0,G0400130010102,"POLYGON ((827801.708 1091626.91, 827865.552 10...",040013001010,3640.0,690331.130975,676962.212041,0.005377
1,G0400130010103,"POLYGON ((765755.318 1002785.693, 765592.367 1...",040013001010,3640.0,690331.130975,9303.984075,0.391230
2,G0400130010104,"POLYGON ((765755.318 1002785.693, 765757.331 1...",040013001010,3640.0,690331.130975,4064.934859,0.895463
3,G0400130030401,"POLYGON ((707726.563 1037891.22, 707726.326 10...",040013003040,2730.0,20964.266666,6558.993303,0.416222
4,G0400130030402,"POLYGON ((707726.563 1037891.22, 707707.434 10...",040013003040,2730.0,20964.266666,14405.273363,0.189514


In [35]:
tracts_with_jobs[["TRACT_GEOID", "jobs_total", "tract_acres", "jobs_per_acre"]].sort_values("jobs_per_acre", ascending=False).head(15)

,TRACT_GEOID,jobs_total,tract_acres,jobs_per_acre
577,040013031910,62678.0,81.690071,767.265833
693,040013042221,92216.0,318.656463,289.390019
691,040013042221,92216.0,320.311472,287.894777
690,040013042221,92216.0,323.011320,285.488446
695,040013042221,92216.0,324.791909,283.923329
694,040013042221,92216.0,334.743528,275.482548
578,040013031910,62678.0,236.493517,265.030520
579,040013031910,62678.0,318.172283,196.993903
689,040013042221,92216.0,558.822090,165.018530
688,040013042221,92216.0,628.734689,146.669178


In [34]:
lodes_jobs_path = JOBS_DIR / "jobs_2022_by_tracts.gpkg"
tracts_with_jobs.to_file(lodes_jobs_path, driver="GPKG")

print("Saved LODES jobs by tracts to:", lodes_jobs_path)

Saved LODES jobs by tracts to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\jobs\lodes_processed\jobs_2022_by_tracts.gpkg
